# Pokemon TCG — card-pool exploration

An interactive look at the **full card pool (1,056 Pokemon printings, 885 unique names)** behind the
competition, built from `pokemon-tcg-ai-battle/EN_Card_Data.csv`:

1. **Energy types** — what the pool is made of
2. **HP vs strongest attack** — with evolution lines drawn on the chart
3. **Themes** — owner-tagged Pokemon (Team Rocket's, Erika's, …) and the Ancient/Future archetypes
4. **Attack efficiency** — Pokemon ranked by energy cost per point of damage
5. **Weakness coverage** — which energy type double-damages the most of the pool,
   and whether the shipped deck (`deck.csv`) can exploit it

All charts are plotly: hover any mark for details, click a legend entry to toggle it,
double-click to isolate it.

In [5]:
import re
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import polars as pl

from tcg.decks import ROOT_DECK_PATH, load_deck_file

pl.Config.set_tbl_rows(25)
px.defaults.template = "plotly_white"
px.defaults.height = 550

In [6]:
# Energy identity colors: canonical TCG hues, tuned until the dataviz palette
# validator passes in this order (= pool-count order, so legend neighbors stay
# CVD-separable). Two residual WARNs are carried by secondary encoding wherever
# the palette is used - Fire<->Lightning CVD 7.9 (floor band) and Colorless
# contrast 2.45:1 - every chart ships axis/direct labels, hover tooltips and
# legend click-to-isolate, so identity is never color-alone.
ENERGY_COLORS = {
    "Grass": "#3c8a2e",
    "Psychic": "#7c3aa6",
    "Water": "#2f86e0",
    "Fighting": "#b05f2a",
    "Darkness": "#5b4f9e",
    "Colorless": "#c29f45",
    "Fire": "#d64550",
    "Lightning": "#bd8500",
    "Metal": "#4577c2",
    "Dragon": "#9b7210",
}
TYPE_ORDER = list(ENERGY_COLORS)
STAGE_ORDER = ["Basic", "Stage 1", "Stage 2"]

ACCENT = "#2a6fb8"        # "deck covers it" bars
DEEMPHASIS = "#b6b6b6"    # "not covered" bars
SEGMENT_GRAY = "rgba(184, 184, 184, 0.35)"  # evolution-line segments under the scatter
MIXED_PARENT = "#e3e3e3"  # treemap tiles whose children span several types


def style_figure(fig: go.Figure, *, title: str) -> go.Figure:
    """Single styling choke point so every chart shares the same look."""
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=17)),
        font=dict(family="Inter, system-ui, sans-serif", size=13, color="#3b3b3b"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        hoverlabel=dict(bgcolor="white", font_size=12),
        margin=dict(l=60, r=30, t=70, b=50),
        legend=dict(title=None),
    )
    fig.update_xaxes(gridcolor="#ececec", zeroline=False)
    fig.update_yaxes(gridcolor="#ececec", zeroline=False)
    return fig

## Loading the pool

The CSV holds **one row per printed move** (attack, ability or Tera marker), so parsing means:
filter to real Pokemon stages (`Pokémon Tool` rows also contain the word "Pokémon"),
translate `{X}`/`竜` type symbols, count `●` colorless slots in attack costs, and split the
`Damage` column into a base number plus a *variable* flag (`30×` multipliers and the two `120-`
style reductions print a number their real output doesn't guarantee).
Owner themes come from the machine-readable `Category` column (`Trainer's Pokémon（Erika）`),
which also tags the **Ancient**/**Future** archetypes.

In [7]:
CSV_PATH = Path("../pokemon-tcg-ai-battle/EN_Card_Data.csv")
STAGE_COLUMN = "Stage (Pokémon)/Type (Energy and Trainer)"
STAGE_NAMES = {"Basic Pokémon": "Basic", "Stage 1 Pokémon": "Stage 1", "Stage 2 Pokémon": "Stage 2"}
SYMBOL_TO_TYPE = {
    "{G}": "Grass", "{R}": "Fire", "{W}": "Water", "{L}": "Lightning", "{P}": "Psychic",
    "{F}": "Fighting", "{D}": "Darkness", "{M}": "Metal", "{C}": "Colorless", "竜": "Dragon",
}
COST_TOKEN = re.compile(r"\{.\}|●")  # ● = a colorless slot, payable by any energy
DAMAGE_PATTERN = re.compile(r"(-?)(\d+)(×?)")
OWNER_PATTERN = re.compile(r"Trainer's Pokémon（(?P<owner>.+)）")
ARCHETYPE_THEMES = ("Ancient", "Future")
_MISSING = ("", "n/a")


def clean(value: str | None) -> str | None:
    """Blank / 'n/a' CSV entries to None."""
    return None if value is None or value.strip() in _MISSING else value.strip()


def parse_energy_type(symbol: str | None) -> str | None:
    symbol = clean(symbol)
    return None if symbol is None else SYMBOL_TO_TYPE[symbol]


def parse_cost(cost: str | None) -> list[str] | None:
    """Attack cost as a list of type names.

    None means "not an attack" (ability and Tera marker rows); the literal
    'No cost' is a real zero-cost attack and parses to [].
    """
    cost = clean(cost)
    if cost is None:
        return None
    if cost == "No cost":
        return []
    return [SYMBOL_TO_TYPE.get(token, "Colorless") for token in COST_TOKEN.findall(cost)]


def parse_damage(damage: str | None) -> dict:
    """Damage cell -> {base, variable}: '130' -> fixed 130; '30×' and '-120'
    (printed '120-') -> variable; blank/'n/a' (effect-only) -> base None."""
    damage = clean(damage)
    if damage is None:
        return {"base": None, "variable": False}
    sign, base, multiplier = DAMAGE_PATTERN.fullmatch(damage).groups()
    return {"base": int(base), "variable": bool(sign or multiplier)}


def owner_theme(category: str | None) -> str | None:
    """Theme from the Category column: the owning trainer, or an archetype."""
    category = clean(category)
    if category is None:
        return None
    if match := OWNER_PATTERN.fullmatch(category):
        return match["owner"]
    return category if category in ARCHETYPE_THEMES else None


def evolution_root(name: str, parent_by_name: dict[str, str | None]) -> str:
    """Walk evolves-from links down to the line's basic. Tolerates parents
    missing from the pool (fossil lines) and defends against cycles."""
    seen = {name}
    while (parent := parent_by_name.get(name)) is not None and parent not in seen:
        name = parent
        seen.add(name)
    return name

In [8]:
def load_move_rows(path: Path = CSV_PATH) -> pl.DataFrame:
    """One tidy row per printed Pokemon move (attack, ability or Tera marker)."""
    return (
        pl.read_csv(path)
        .filter(pl.col(STAGE_COLUMN).is_in(list(STAGE_NAMES)))
        .select(
            card_id=pl.col("Card ID"),
            name=pl.col("Card Name").str.strip_chars(),
            expansion=pl.col("Expansion"),
            stage=pl.col(STAGE_COLUMN).replace_strict(STAGE_NAMES),
            rule=pl.col("Rule").map_elements(clean, return_dtype=pl.String),
            category=pl.col("Category"),
            evolves_from=pl.col("Previous stage").map_elements(clean, return_dtype=pl.String),
            hp=pl.col("HP").cast(pl.Int64),
            energy_type=pl.col("Type").map_elements(parse_energy_type, return_dtype=pl.String),
            weakness=pl.col("Weakness").map_elements(parse_energy_type, return_dtype=pl.String),
            resistance=pl.col("Resistance (Type)").map_elements(parse_energy_type, return_dtype=pl.String),
            retreat=pl.col("Retreat").cast(pl.Int64, strict=False),
            move_name=pl.col("Move Name").map_elements(clean, return_dtype=pl.String),
            cost=pl.col("Cost").map_elements(parse_cost, return_dtype=pl.List(pl.String)),
            damage_raw=pl.col("Damage"),
        )
        .with_columns(
            pl.col("damage_raw")
            .map_elements(parse_damage, return_dtype=pl.Struct({"base": pl.Int64, "variable": pl.Boolean}))
            .alias("parsed")
        )
        .unnest("parsed")
        .rename({"base": "damage", "variable": "variable_damage"})
        .with_columns(pl.col("variable_damage").fill_null(False))
        .drop("damage_raw")
    )


def build_attacks_df(moves: pl.DataFrame) -> pl.DataFrame:
    """One row per printed attack (ability/Tera rows have a null cost)."""
    return (
        moves.filter(pl.col("cost").is_not_null())
        .with_columns(
            cost_size=pl.col("cost").list.len(),
            cost_str=pl.when(pl.col("cost").list.len() == 0)
            .then(pl.lit("Free"))
            .otherwise(pl.col("cost").list.join(", ")),
        )
        .with_columns(
            # Energy attachments per point of damage; defined only for fixed,
            # positive damage. A free damaging attack is legitimately 0.
            cost_per_damage=pl.when((pl.col("damage") > 0) & ~pl.col("variable_damage"))
            .then(pl.col("cost_size") / pl.col("damage"))
        )
        .select(
            "card_id", "name", "energy_type", "stage", "move_name",
            "cost_size", "cost_str", "damage", "variable_damage", "cost_per_damage",
        )
    )


def build_pokemon_df(moves: pl.DataFrame, attacks: pl.DataFrame) -> pl.DataFrame:
    """One row per Pokemon printing, with its attacks aggregated."""
    cards = moves.group_by("card_id").agg(
        pl.col(
            "name", "expansion", "stage", "rule", "category", "evolves_from",
            "hp", "energy_type", "weakness", "resistance", "retreat",
        ).first()
    )
    per_card = attacks.group_by("card_id").agg(
        n_attacks=pl.len(),
        max_damage=pl.col("damage").filter(~pl.col("variable_damage")).max(),
        has_variable_attack=pl.col("variable_damage").any(),
        best_cost_per_damage=pl.col("cost_per_damage").min(),
    )
    pokemon = (
        cards.join(per_card, on="card_id", how="left")
        .with_columns(
            pl.col("n_attacks").fill_null(0),
            pl.col("max_damage").fill_null(0),
            pl.col("has_variable_attack").fill_null(False),
            owner=pl.col("category").map_elements(owner_theme, return_dtype=pl.String),
        )
    )
    links = pokemon.select("name", "evolves_from").unique()
    assert links.height == pokemon["name"].n_unique(), "printings of one name disagree on evolves_from"
    parent_by_name = dict(links.iter_rows())
    return pokemon.with_columns(
        family_root=pl.col("name").map_elements(
            lambda name: evolution_root(name, parent_by_name), return_dtype=pl.String
        )
    ).sort("card_id")


def best_printing_per_name(pokemon: pl.DataFrame) -> pl.DataFrame:
    """Collapse reprints: per name, keep the printing with the highest (hp, max_damage)."""
    return (
        pokemon.sort(["hp", "max_damage"], descending=True)
        .unique(subset="name", keep="first", maintain_order=True)
    )


moves = load_move_rows()
attacks_df = build_attacks_df(moves)
pokemon_df = build_pokemon_df(moves, attacks_df)

assert pokemon_df.height == 1_056, pokemon_df.height
assert set(pokemon_df["energy_type"]) == set(TYPE_ORDER)
assert pokemon_df["hp"].null_count() == 0
assert pokemon_df["family_root"].null_count() == 0
assert attacks_df["move_name"].null_count() == 0
assert attacks_df.filter(pl.col("cost_size") > 0).height + attacks_df.filter(pl.col("cost_str") == "Free").height == attacks_df.height

print(f"{pokemon_df.height} Pokemon printings, {pokemon_df['name'].n_unique()} unique names, "
      f"{attacks_df.height} attacks")

FileNotFoundError: The system cannot find the file specified. (os error 2): ..\pokemon-tcg-ai-battle\EN_Card_Data.csv

## 1 · Energy types in the pool

Every Pokemon card has exactly one energy type — the type its attacks count as, and the type
weakness math checks. Grass is the biggest slice of the pool, Dragon by far the smallest.

In [ ]:
type_counts = (
    pokemon_df.group_by("energy_type").agg(n_cards=pl.len()).sort("n_cards", descending=True)
)

fig = px.bar(
    type_counts,
    x="energy_type", y="n_cards", text="n_cards",
    color="energy_type", color_discrete_map=ENERGY_COLORS,
    category_orders={"energy_type": TYPE_ORDER},
    labels={"energy_type": "", "n_cards": "Pokemon cards"},
)
fig.update_traces(textposition="outside", width=0.65, showlegend=False)
fig.update_xaxes(categoryorder="total descending")
style_figure(fig, title="Pokemon in the card pool by energy type")

## 2 · HP vs strongest attack, with evolution lines

Each point is a unique Pokemon name (reprints collapsed to the strongest printing).
**x** = its biggest *fixed-damage* attack, **y** = HP; marker shape encodes the evolution stage
and the gray segments connect each Pokemon to its pre-evolution, so whole evolution lines read
as little upward paths. The column at **x = 0** is Pokemon whose attacks are all
variable (`30×`-style) or pure effects — their punch isn't printed as a fixed number.

*Tip: double-click a type in the legend to isolate it; hover for the line, weakness and best attack.*

In [ ]:
def evolution_segments(points: pl.DataFrame) -> go.Scatter:
    """One None-separated line trace joining each evolution to its pre-evolution."""
    parents = points.select(
        parent=pl.col("name"),
        parent_damage=pl.col("max_damage"),
        parent_hp=pl.col("hp"),
    )
    pairs = (
        points.drop_nulls("evolves_from")
        .join(parents, left_on="evolves_from", right_on="parent", how="inner")
    )
    xs: list[int | None] = []
    ys: list[int | None] = []
    for parent_damage, parent_hp, child_damage, child_hp in pairs.select(
        "parent_damage", "parent_hp", "max_damage", "hp"
    ).iter_rows():
        xs += [parent_damage, child_damage, None]
        ys += [parent_hp, child_hp, None]
    return go.Scatter(
        x=xs, y=ys, mode="lines",
        line=dict(color=SEGMENT_GRAY, width=1),
        hoverinfo="skip", showlegend=False,
    )


strongest_attacks = (
    attacks_df.filter(~pl.col("variable_damage"))
    .sort("damage", descending=True, nulls_last=True)
    .unique(subset="card_id", keep="first", maintain_order=True)
    .select("card_id", best_attack=pl.col("move_name"), best_attack_cost=pl.col("cost_str"))
)
pool = best_printing_per_name(pokemon_df).join(strongest_attacks, on="card_id", how="left")

fig = px.scatter(
    pool,
    x="max_damage", y="hp",
    color="energy_type", color_discrete_map=ENERGY_COLORS,
    symbol="stage", symbol_map={"Basic": "circle", "Stage 1": "diamond", "Stage 2": "square"},
    category_orders={"energy_type": TYPE_ORDER, "stage": STAGE_ORDER},
    hover_name="name",
    hover_data={
        "energy_type": False, "stage": True, "family_root": True,
        "weakness": True, "best_attack": True, "best_attack_cost": True,
    },
    labels={
        "max_damage": "Strongest fixed-damage attack", "hp": "HP",
        "stage": "stage", "family_root": "evolution line", "weakness": "weak to",
        "best_attack": "best attack", "best_attack_cost": "its cost",
    },
    opacity=0.85,
)
fig.update_traces(
    marker=dict(size=8, line=dict(width=1, color="white")),  # white ring separates overlaps
    selector=dict(mode="markers"),
)
fig.add_trace(evolution_segments(pool))
fig.data = fig.data[-1:] + fig.data[:-1]  # draw segments under the points

shown_types: set[str] = set()  # one legend entry per type; click toggles the whole type
for trace in fig.data:
    if trace.mode == "lines":
        continue
    energy = trace.name.split(",")[0]
    trace.update(legendgroup=energy, name=energy, showlegend=energy not in shown_types)
    shown_types.add(energy)
fig.add_annotation(
    text="marker shape:  ● Basic   ◆ Stage 1   ■ Stage 2",
    xref="paper", yref="paper", x=1, y=1.04, showarrow=False,
    font=dict(size=12, color="#6b6b6b"),
)
fig.update_layout(height=680)
style_figure(fig, title="HP vs strongest attack — gray segments join evolution lines")

## 3 · Thematic clusters

The `Category` column tags **owner themes** (a trainer's own Pokemon, like *Team Rocket's Mewtwo ex*
or *Erika's Vileplume ex*) plus the **Ancient** and **Future** paradox archetypes.
Tile color is the Pokemon's energy type, so a theme's type profile is visible at a glance —
mixed-type themes get gray group tiles, single-type themes (Misty = Water, Erika = Grass) light up.

In [ ]:
themed = (
    best_printing_per_name(pokemon_df)
    .filter(pl.col("owner").is_not_null())
    .with_columns(n=pl.lit(1))
)

fig = px.treemap(
    themed,
    path=[px.Constant("All themes"), "owner", "name"],
    values="n",
    color="energy_type",
    color_discrete_map={**ENERGY_COLORS, "(?)": MIXED_PARENT},
    hover_data={"hp": True, "stage": True, "weakness": True},
)
fig.update_traces(marker_line=dict(color="white", width=2), textfont_size=13)
fig.update_layout(height=620)
style_figure(fig, title="Owner and archetype themes — tile color = energy type")

In [ ]:
theme_summary = (
    themed.group_by("owner")
    .agg(
        n_pokemon=pl.len(),
        avg_hp=pl.col("hp").mean().round(0).cast(pl.Int64),
        dominant_type=pl.col("energy_type").mode().sort().first(),
        stage_2_lines=(pl.col("stage") == "Stage 2").sum(),
    )
    .sort("n_pokemon", descending=True)
)
theme_summary

## 4 · Attack efficiency — energy per point of damage

**Metric.** For each attack with a *fixed, positive* printed damage:
`cost_per_damage = energy slots / damage` (a `●` colorless slot still costs an attachment, so it
counts as 1). Each Pokemon name is then ranked by its **cheapest** such attack — the minimum, not
the mean, because in a game you simply use your best attack. Shown below as
**energy per 100 damage** (lower = better). Variable (`30×`) and pure-effect attacks are excluded;
names with no rankable attack at all are dropped and counted in the printout.

In [ ]:
efficiency = (
    attacks_df.drop_nulls("cost_per_damage")
    .sort("cost_per_damage")
    .unique(subset="name", keep="first", maintain_order=True)
    .with_columns(energy_per_100=(pl.col("cost_per_damage") * 100).round(2))
)
n_unranked = pokemon_df["name"].n_unique() - efficiency["name"].n_unique()
print(f"{efficiency.height} Pokemon ranked; {n_unranked} have no fixed-damage attack and are excluded")


def efficiency_bar(rows: pl.DataFrame, *, title: str, best_on_top: bool = True) -> go.Figure:
    # color= splits the bars into per-type traces, which would scramble the
    # y-axis order - pin it explicitly, bottom to top.
    ordered = rows.sort("energy_per_100", descending=best_on_top)
    fig = px.bar(
        ordered,
        x="energy_per_100", y="name", orientation="h", text="energy_per_100",
        color="energy_type", color_discrete_map=ENERGY_COLORS,
        category_orders={"energy_type": TYPE_ORDER},
        hover_data={"move_name": True, "cost_str": True, "damage": True, "energy_type": False},
        labels={
            "energy_per_100": "Energy per 100 damage (lower = better)", "name": "",
            "move_name": "attack", "cost_str": "cost", "damage": "damage",
        },
    )
    fig.update_traces(textposition="outside", width=0.65)
    fig.update_yaxes(categoryorder="array", categoryarray=ordered["name"].to_list())
    fig.update_layout(height=620)
    return style_figure(fig, title=title)


efficiency_bar(efficiency.head(20), title="Most efficient attackers in the pool")

In [ ]:
efficiency_bar(efficiency.tail(20), title="Least efficient attackers in the pool", best_on_top=False)

## 5 · Weakness coverage — and does our deck exploit it?

A Pokemon takes **double damage** from attackers whose *own energy type* matches its printed
weakness (the engine checks the attacker's type, not the attack's cost — `tcg/models.py`).
So the question "which energy type has the biggest double-damage coverage of the pool?" is:
count Pokemon cards weak to each type, then check whether the shipped `deck.csv` fields an
attacker of that type.

In [ ]:
deck_pokemon = (
    pokemon_df.filter(pl.col("card_id").is_in(load_deck_file(ROOT_DECK_PATH)))
    .filter(pl.col("n_attacks") > 0)
    .unique(subset="name")
)
attackers_by_type = {
    energy: ", ".join(sorted(names))
    for energy, names in deck_pokemon.group_by("energy_type").agg("name").iter_rows()
}

weakness_coverage = (
    pokemon_df.drop_nulls("weakness")
    .group_by("weakness")
    .agg(n_weak=pl.len())
    .sort("n_weak", descending=True)
    .with_columns(
        deck_attackers=pl.col("weakness").replace_strict(attackers_by_type, default="—"),
        covered=pl.col("weakness").is_in(list(attackers_by_type)),
    )
    .with_columns(
        status=pl.when(pl.col("covered")).then(pl.lit("deck covers it")).otherwise(pl.lit("not covered")),
        marker=pl.when(pl.col("covered")).then(pl.lit("deck ✓")).otherwise(pl.lit("")),
    )
)

fig = px.bar(
    weakness_coverage,
    x="n_weak", y="weakness", orientation="h", text="marker",
    color="status", color_discrete_map={"deck covers it": ACCENT, "not covered": DEEMPHASIS},
    hover_data={"deck_attackers": True, "status": False},
    labels={
        "n_weak": "Pokemon cards weak to this type (of 1,056)", "weakness": "",
        "deck_attackers": "deck attackers",
    },
)
fig.update_traces(textposition="outside", width=0.65)
fig.update_yaxes(categoryorder="total ascending")
style_figure(fig, title="Double-damage coverage of the pool, by attacking energy type")

## Takeaways

- **Fire is the most exploitable energy type**: 220 of 1,056 Pokemon cards (21%) take double
  damage from Fire attackers, ahead of Fighting (188) and Lightning (155). The shipped mono-Water
  deck (Snover / Mega Abomasnow ex / Kyogre) covers only **Water weakness — rank 5 of 8, 99 cards**.
  A Fire or Fighting attacker would double-damage roughly twice as much of the pool.
- **Free damage exists**: Budew, Tyrogue and Ethan's Pichu attack for zero energy;
  the best *paid* attack in the pool is Palafin ex's **Giga Impact — 250 damage for 1 energy**.
  At the other end, Wailord's Hydro Pump pays 4 energy for 10 damage.
- **Team Rocket is by far the biggest theme** (52 Pokemon and mixed across types),
  while classic gym-leader themes (Misty = Water, Erika = Grass) are small and mono-type.
- Evolution lines read as up-and-right paths in §2: evolving buys HP and damage at once,
  with Stage 2 / Mega lines topping out around 340 HP.

**Caveats.** Variable-damage attacks (110 `×` multipliers, 2 damage-reducers) and pure-effect
attacks carry no fixed number, so §2 places those Pokemon at x = 0 and §4 skips them (72 names
have no rankable attack). Reprints are collapsed to the strongest printing in §2–§4;
§1 and §5 count every printing in the pool.